# CRISE-ID — Personal Digital Forensics Demo

This notebook answers one question for your own face:
> *Which regions does ArcFace actually rely on to identify you?*

CRISE-ID produces a saliency map that reflects the **competitive 1:N decision structure** — not just similarity to yourself, but how much a masked region lets the model distinguish you from 1,680 other identities.

**Workflow:**
1. Place 5–10 photos of your face in `data/demo_identity/{YOUR_NAME}/`
2. Cell 1 enrolls you (first photo → gallery); remaining photos are probes
3. CRISE runs on each probe
4. Four forensics panels:
   - **Panel A** — Saliency overview: chip / map / overlay per probe + mean
   - **Panel B** — Per-region importance: how much of the total saliency mass falls in each of 5 facial zones
   - **Panel C** — Deletion experiment: recognition confidence vs. fraction of most-important pixels masked
   - **Panel D** — Cross-probe consistency: are your maps stable across photos?

All outputs are saved under `results/demo_forensics/{YOUR_NAME}/`.

In [ ]:
# ---------------------------------------------------------------------------
# Config — SET YOUR_NAME BEFORE RUNNING
# ---------------------------------------------------------------------------

YOUR_NAME     = "your_name_here"   # ← no spaces, use underscore
PHOTO_DIR     = f"data/demo_identity/{YOUR_NAME}"

GALLERY_EMB   = "cache/G.npy"
GALLERY_IDS   = "cache/gallery_ids.npy"
OUT_DIR       = f"results/demo_forensics/{YOUR_NAME}"
CRISE_OUT_DIR = f"{OUT_DIR}/crise_maps"

# CRISE hyperparams
N        = 1000
S        = 8
P1       = 0.5
TAU      = 0.1
RUN_SEED = 9999

# Deletion experiment pixel budgets (fraction of 112×112)
BUDGETS  = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis

from rise import build_aligned_chip_112, get_embedding_from_chip
from crise import run_crise

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CRISE_OUT_DIR, exist_ok=True)

assert os.path.isdir(PHOTO_DIR), (
    f"Photo directory not found: {PHOTO_DIR}\n"
    f"Create it and add 5-10 photos of your face."
)

print(f"Output dir : {OUT_DIR}")
print(f"CRISE maps : {CRISE_OUT_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Enrollment — first photo is gallery, rest are probes
# ---------------------------------------------------------------------------

photo_paths = sorted([
    os.path.join(PHOTO_DIR, f)
    for f in os.listdir(PHOTO_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
assert len(photo_paths) >= 2, "Need at least 2 photos (1 gallery + 1 probe)"

gallery_path = photo_paths[0]
probe_paths  = photo_paths[1:]

print(f"Gallery : {os.path.basename(gallery_path)}")
print(f"Probes  : {len(probe_paths)} images")

# Load base gallery and augment with personal embedding (in-memory only)
G_base      = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()

gallery_img  = cv2.imread(gallery_path)
gallery_chip = build_aligned_chip_112(gallery_img, app)
your_emb     = get_embedding_from_chip(gallery_chip, rec).reshape(1, -1)

G_demo      = np.vstack([G_base, your_emb]).astype(np.float32)
ids_demo    = gallery_ids + [YOUR_NAME]
id_to_index = {gid: i for i, gid in enumerate(ids_demo)}

print(f"\nGallery size: {G_base.shape[0]} → {G_demo.shape[0]} (added {YOUR_NAME})")
print(f"Your embedding norm: {np.linalg.norm(your_emb):.4f}")

# Self-match sanity check
sims     = G_demo @ your_emb.ravel()
rank1_id = ids_demo[int(np.argmax(sims))]
print(f"Gallery self-match : {rank1_id}  (expected: {YOUR_NAME})")

In [ ]:
# ---------------------------------------------------------------------------
# Verify probe images match YOU at rank-1 before analysis
# ---------------------------------------------------------------------------

probe_results = []
for path in probe_paths:
    img = cv2.imread(path)
    if img is None:
        print(f"[skip] could not read {path}")
        continue
    try:
        chip  = build_aligned_chip_112(img, app)
        emb   = get_embedding_from_chip(chip, rec)
        sims  = G_demo @ emb
        rank1 = ids_demo[int(np.argmax(sims))]
        sim   = float(sims[id_to_index[YOUR_NAME]])
        probe_results.append({
            "path": path, "rank1": rank1, "sim": sim,
            "correct": rank1 == YOUR_NAME, "chip": chip, "emb": emb
        })
        status = "OK" if rank1 == YOUR_NAME else "FAIL"
        print(f"  [{status}] {os.path.basename(path):30s}  sim={sim:.3f}  rank1={rank1}")
    except Exception as e:
        print(f"[error] {path}: {e}")

n_correct = sum(r["correct"] for r in probe_results)
print(f"\nBaseline rank-1: {n_correct}/{len(probe_results)} probes correct")

In [ ]:
# ---------------------------------------------------------------------------
# Run CRISE on all probe images
# (already-computed maps are loaded from cache; re-running is idempotent)
# ---------------------------------------------------------------------------

crise_results = []
for k, pr in enumerate(tqdm(probe_results, desc="CRISE")):
    try:
        result = run_crise(
            true_id     = YOUR_NAME,
            probe_path  = pr["path"],
            G           = G_demo,
            id_to_index = id_to_index,
            rec         = rec,
            app         = app,
            run_seed    = RUN_SEED + k,
            out_dir     = CRISE_OUT_DIR,
            tau         = TAU,
            N           = N,
            s           = S,
            p1          = P1,
        )
        crise_results.append(result)
    except Exception as e:
        print(f"[fail] {pr['path']}: {e}")

print(f"\nCRISE complete: {len(crise_results)} / {len(probe_results)} maps")

sal_maps = [r["sal_norm"] for r in crise_results if r["sal_norm"] is not None]
mean_sal = np.mean(sal_maps, axis=0).astype(np.float32)

np.save(os.path.join(OUT_DIR, f"{YOUR_NAME}_mean_saliency.npy"), mean_sal)
print(f"Mean saliency map saved.")

In [ ]:
# ---------------------------------------------------------------------------
# Panel A — Saliency Overview
# Chip / saliency map / overlay for each probe, plus mean map
# ---------------------------------------------------------------------------

n_probes = len(crise_results)
n_cols   = 3
n_rows   = n_probes + 1   # one row per probe + mean row

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
if n_rows == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Aligned chip", "CRISE saliency", "Overlay"]
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10, fontweight="bold")

for i, r in enumerate(crise_results):
    chip_rgb = cv2.cvtColor(r["chip_bgr"], cv2.COLOR_BGR2RGB)
    sal      = r["sal_norm"]
    fname    = os.path.basename(r["probe_path"])

    axes[i, 0].imshow(chip_rgb)
    axes[i, 0].set_ylabel(fname, fontsize=7, rotation=0, labelpad=70, va="center")

    axes[i, 1].imshow(sal, cmap="inferno", vmin=0, vmax=1)

    axes[i, 2].imshow(chip_rgb)
    axes[i, 2].imshow(sal, alpha=0.55, cmap="inferno", vmin=0, vmax=1)
    axes[i, 2].set_title(f"sim={r['w_clean']:.3f}", fontsize=8)

# Mean row
ref_chip_rgb = cv2.cvtColor(crise_results[0]["chip_bgr"], cv2.COLOR_BGR2RGB)
axes[-1, 0].imshow(ref_chip_rgb)
axes[-1, 0].set_ylabel("MEAN", fontsize=8, rotation=0, labelpad=70, va="center", fontweight="bold")
axes[-1, 1].imshow(mean_sal, cmap="inferno", vmin=0, vmax=1)
axes[-1, 2].imshow(ref_chip_rgb)
axes[-1, 2].imshow(mean_sal, alpha=0.55, cmap="inferno", vmin=0, vmax=1)

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(f"Panel A — CRISE Saliency Overview: {YOUR_NAME}", fontsize=12, y=1.01)
plt.tight_layout()
path_a = os.path.join(OUT_DIR, "panel_A_saliency_overview.png")
plt.savefig(path_a, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {path_a}")

In [ ]:
# ---------------------------------------------------------------------------
# Panel B — Per-Region Importance
#
# 5 anatomical zones defined on the 112×112 ArcFace-aligned chip.
# Zones are calibrated to the standard ArcFace 5-point landmark destinations:
#   left_eye  (~38, 52)   right_eye (~74, 52)
#   nose_tip  (~56, 72)
#   left_mouth(~42, 92)   right_mouth(~71, 92)
# ---------------------------------------------------------------------------

def make_region_masks_112() -> dict:
    """Return dict of region_name -> (112,112) boolean mask."""
    masks = {}
    m = np.zeros((112, 112), dtype=bool)

    # Forehead / upper face (rows 5-38, full width)
    m[:] = False;  m[5:38, 12:100]  = True;  masks["Forehead"]   = m.copy()

    # Eye zone — spans both eyes and brow ridge (rows 35-62)
    m[:] = False;  m[35:62, 12:100] = True;  masks["Eye zone"]   = m.copy()

    # Nose zone (rows 58-82, columns 28-84)
    m[:] = False;  m[58:82, 28:84]  = True;  masks["Nose"]       = m.copy()

    # Mouth / perioral zone (rows 78-100, columns 22-90)
    m[:] = False;  m[78:100, 22:90] = True;  masks["Mouth"]      = m.copy()

    # Jaw / lower face (rows 97-112, columns 10-102)
    m[:] = False;  m[97:112, 10:102]= True;  masks["Jaw/chin"]   = m.copy()

    return masks


REGION_MASKS  = make_region_masks_112()
REGION_NAMES  = list(REGION_MASKS.keys())
REGION_COLORS = ["#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F"]


def region_importance(sal: np.ndarray, masks: dict) -> dict:
    """Fractional saliency mass in each region (sums to ≤1 since regions can overlap)."""
    total = sal.sum() + 1e-9
    return {name: float(sal[mask].sum() / total) for name, mask in masks.items()}


# Per-probe region importance
per_probe_ri = [region_importance(r["sal_norm"], REGION_MASKS) for r in crise_results]
mean_ri      = region_importance(mean_sal, REGION_MASKS)

# ---- Figure: grouped bar chart ----
probe_labels = [os.path.basename(r["probe_path"]) for r in crise_results]
x            = np.arange(len(REGION_NAMES))
width        = 0.7 / (len(crise_results) + 1)

fig, ax = plt.subplots(figsize=(10, 4.5))

for pi, (ri, label) in enumerate(zip(per_probe_ri, probe_labels)):
    vals   = [ri[r] for r in REGION_NAMES]
    offset = (pi - len(crise_results) / 2) * width
    ax.bar(x + offset, vals, width, label=label, alpha=0.65)

# Mean as bold line
mean_vals = [mean_ri[r] for r in REGION_NAMES]
ax.plot(x, mean_vals, "ko-", linewidth=2, markersize=6, zorder=5, label="Mean")

ax.set_xticks(x)
ax.set_xticklabels(REGION_NAMES, fontsize=11)
ax.set_ylabel("Fractional saliency mass", fontsize=10)
ax.set_title(f"Panel B — Per-Region Importance: {YOUR_NAME}", fontsize=12)
ax.legend(fontsize=8, loc="upper right")
ax.set_ylim(0, None)
ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
path_b = os.path.join(OUT_DIR, "panel_B_region_importance.png")
plt.savefig(path_b, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {path_b}")

# Print region table
ri_df = pd.DataFrame(per_probe_ri, index=probe_labels)
ri_df.loc["MEAN"] = mean_ri
print("\nFractional saliency mass per region:")
print(ri_df.round(3).to_string())

top_region = max(mean_ri, key=mean_ri.get)
print(f"\nMost salient region (mean): {top_region}  ({mean_ri[top_region]:.3f})")

In [ ]:
# ---------------------------------------------------------------------------
# Panel B (supplemental) — Region mask overlay on mean chip
# Shows where the 5 zones are on the face
# ---------------------------------------------------------------------------

ref_chip = crise_results[0]["chip_bgr"]
ref_rgb  = cv2.cvtColor(ref_chip, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(REGION_NAMES) + 1, figsize=(3 * (len(REGION_NAMES) + 1), 3))

axes[0].imshow(ref_rgb)
axes[0].set_title("Reference chip", fontsize=9)

for i, (name, mask) in enumerate(REGION_MASKS.items()):
    overlay = ref_rgb.copy().astype(float)
    color   = matplotlib.colors.to_rgb(REGION_COLORS[i])
    for c in range(3):
        overlay[..., c][mask] = overlay[..., c][mask] * 0.4 + color[c] * 255 * 0.6
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    ri_val  = mean_ri[name]
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f"{name}\n{ri_val:.3f}", fontsize=9)

for ax in axes:
    ax.axis("off")

plt.suptitle(f"Panel B — Region Zones + Mean Importance: {YOUR_NAME}", fontsize=11)
plt.tight_layout()
path_b2 = os.path.join(OUT_DIR, "panel_B_region_zones.png")
plt.savefig(path_b2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {path_b2}")

In [ ]:
# ---------------------------------------------------------------------------
# Panel C — Deletion Experiment
#
# Progressively mask the top-k% most salient pixels with the mean face colour.
# Measure: ArcFace similarity to YOU and rank-1 correctness at each budget.
# Three conditions:
#   1. CRISE-guided   (high-saliency pixels first)
#   2. Random order   (pixel shuffle)
#   3. Reverse-CRISE  (low-saliency pixels first — control)
# ---------------------------------------------------------------------------

FINE_BUDGETS = np.linspace(0, 0.6, 31)   # 0% to 60% in 2% steps
YOUR_IDX     = id_to_index[YOUR_NAME]

def run_deletion(chip_bgr, sal_map, G, your_idx, rng_seed=42):
    """Return three arrays: crise_sims, random_sims, reverse_sims across FINE_BUDGETS."""
    mean_color = chip_bgr.mean(axis=(0, 1))
    flat_sal   = sal_map.ravel()
    n_pixels   = flat_sal.size

    order_crise   = np.argsort(flat_sal)[::-1]   # high → low
    order_reverse = np.argsort(flat_sal)           # low  → high
    rng            = np.random.default_rng(rng_seed)
    order_random   = rng.permutation(n_pixels)

    def sim_at_budget(order, budget):
        k     = int(n_pixels * budget)
        chip  = chip_bgr.copy().astype(np.float32)
        ys, xs = np.unravel_index(order[:k], (112, 112))
        chip[ys, xs] = mean_color
        emb   = get_embedding_from_chip(chip.astype(np.uint8), rec)
        return float(G[your_idx] @ emb)

    crise_sims   = [sim_at_budget(order_crise,   b) for b in FINE_BUDGETS]
    random_sims  = [sim_at_budget(order_random,  b) for b in FINE_BUDGETS]
    reverse_sims = [sim_at_budget(order_reverse, b) for b in FINE_BUDGETS]
    return crise_sims, random_sims, reverse_sims


# Run on each probe and average
all_crise, all_random, all_reverse = [], [], []
for r in tqdm(crise_results, desc="Deletion curves"):
    c, rnd, rv = run_deletion(r["chip_bgr"], r["sal_norm"], G_demo, YOUR_IDX)
    all_crise.append(c)
    all_random.append(rnd)
    all_reverse.append(rv)

mean_crise   = np.mean(all_crise,   axis=0)
mean_random  = np.mean(all_random,  axis=0)
mean_reverse = np.mean(all_reverse, axis=0)

# ---- Figure ----
fig, ax = plt.subplots(figsize=(8, 4.5))

pct = FINE_BUDGETS * 100
ax.plot(pct, mean_crise,   "r-o",  markersize=4, linewidth=2, label="CRISE-guided (↑saliency first)")
ax.plot(pct, mean_random,  "k--s", markersize=4, linewidth=1.5, label="Random order")
ax.plot(pct, mean_reverse, "b-^",  markersize=4, linewidth=1.5, label="Reverse-CRISE (↓saliency first)")

# Mark rank-1 threshold (typically ~0.28 for ArcFace on LFW)
RANK1_THRESH = 0.28
ax.axhline(RANK1_THRESH, color="gray", linestyle=":", linewidth=1.2, label=f"Rank-1 threshold (~{RANK1_THRESH})")

ax.set_xlabel("% of pixels masked (mean face color)", fontsize=11)
ax.set_ylabel("ArcFace similarity to you", fontsize=11)
ax.set_title(f"Panel C — Deletion Experiment: {YOUR_NAME}", fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 60)

plt.tight_layout()
path_c = os.path.join(OUT_DIR, "panel_C_deletion.png")
plt.savefig(path_c, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {path_c}")

# Find budget at which CRISE-guided drops below rank-1 threshold
below = np.where(mean_crise < RANK1_THRESH)[0]
if len(below):
    drop_pct = float(FINE_BUDGETS[below[0]] * 100)
    print(f"\nCRISE-guided masking breaks rank-1 at ~{drop_pct:.0f}% of pixels")
else:
    print("\nCRISE-guided masking did not break rank-1 within 60% budget")

In [ ]:
# ---------------------------------------------------------------------------
# Panel C (supplemental) — Visual strip of masked chips at key budgets
# Shows what your face looks like at each masking level
# ---------------------------------------------------------------------------

VIS_BUDGETS  = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]
ref_r        = crise_results[0]
ref_chip     = ref_r["chip_bgr"]
ref_sal      = mean_sal
mean_color   = ref_chip.mean(axis=(0, 1))
flat_sal     = ref_sal.ravel()
order_crise  = np.argsort(flat_sal)[::-1]

fig, axes = plt.subplots(1, len(VIS_BUDGETS), figsize=(3 * len(VIS_BUDGETS), 3))

for i, b in enumerate(VIS_BUDGETS):
    k = int(112 * 112 * b)
    chip = ref_chip.copy().astype(np.float32)
    if k > 0:
        ys, xs = np.unravel_index(order_crise[:k], (112, 112))
        chip[ys, xs] = mean_color
    chip = chip.astype(np.uint8)

    emb  = get_embedding_from_chip(chip, rec)
    sim  = float(G_demo[YOUR_IDX] @ emb)

    axes[i].imshow(cv2.cvtColor(chip, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{int(b*100)}% masked\nsim={sim:.3f}", fontsize=9)
    axes[i].axis("off")

plt.suptitle(f"Panel C — Masked chips (CRISE-guided): {YOUR_NAME}", fontsize=11)
plt.tight_layout()
path_c2 = os.path.join(OUT_DIR, "panel_C_masked_chips.png")
plt.savefig(path_c2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {path_c2}")

In [ ]:
# ---------------------------------------------------------------------------
# Panel D — Cross-Probe Consistency
#
# Pairwise cosine similarity between all CRISE maps for your probes.
# High consistency = the maps are stable, not noise — evidence that CRISE
# is capturing a genuine identity-specific signal.
# ---------------------------------------------------------------------------

n = len(sal_maps)

if n >= 2:
    # Flatten maps for cosine similarity
    flat_maps = np.array([s.ravel() for s in sal_maps], dtype=np.float32)
    norms     = np.linalg.norm(flat_maps, axis=1, keepdims=True) + 1e-9
    normed    = flat_maps / norms
    cos_sim   = normed @ normed.T   # (n, n)

    # Off-diagonal mean
    mask_offdiag   = ~np.eye(n, dtype=bool)
    mean_cos       = float(cos_sim[mask_offdiag].mean())
    std_cos        = float(cos_sim[mask_offdiag].std())

    # ---- Heatmap ----
    probe_short = [os.path.splitext(os.path.basename(r["probe_path"]))[0] for r in crise_results]

    fig, ax = plt.subplots(figsize=(max(4, n + 1), max(4, n + 1)))
    im = ax.imshow(cos_sim, vmin=0, vmax=1, cmap="YlOrRd")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(n)); ax.set_xticklabels(probe_short, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n)); ax.set_yticklabels(probe_short, fontsize=8)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{cos_sim[i,j]:.2f}", ha="center", va="center", fontsize=8)

    ax.set_title(
        f"Panel D — Cross-Probe Map Consistency: {YOUR_NAME}\n"
        f"Mean off-diag cosine sim = {mean_cos:.3f} ± {std_cos:.3f}",
        fontsize=11
    )
    plt.tight_layout()
    path_d = os.path.join(OUT_DIR, "panel_D_consistency.png")
    plt.savefig(path_d, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path_d}")
    print(f"\nMean pairwise cosine similarity: {mean_cos:.3f} ± {std_cos:.3f}")
    if mean_cos >= 0.90:
        print("Interpretation: Maps are highly stable across probes (>=0.90).")
    elif mean_cos >= 0.75:
        print("Interpretation: Moderate stability. Some variation across lighting/pose.")
    else:
        print("Interpretation: Low stability. Check probe image quality or face alignment.")
else:
    print("Only one saliency map — skipping consistency analysis.")

In [ ]:
# ---------------------------------------------------------------------------
# Forensics Report — Text Summary
# ---------------------------------------------------------------------------

top_two    = sorted(mean_ri, key=mean_ri.get, reverse=True)[:2]
bottom_two = sorted(mean_ri, key=mean_ri.get)[:2]

print("=" * 60)
print(f" CRISE-ID FORENSICS REPORT — {YOUR_NAME}")
print("=" * 60)
print(f"  Gallery size           : {G_demo.shape[0]} identities")
print(f"  Probe images analyzed  : {len(crise_results)}")
print(f"  CRISE params           : N={N}, s={S}, p={P1}, tau={TAU}")
print()
print("IDENTITY VERIFICATION")
clean_sims = [r["w_clean"] for r in crise_results]
print(f"  ArcFace similarity     : {np.mean(clean_sims):.3f} ± {np.std(clean_sims):.3f} (probe mean ± std)")
print(f"  Rank-1 correct         : {n_correct}/{len(probe_results)} probes")
print()
print("SALIENCY — WHICH REGIONS DOES ARCFACE USE TO IDENTIFY YOU?")
for name in sorted(mean_ri, key=mean_ri.get, reverse=True):
    bar  = "█" * int(mean_ri[name] * 40)
    print(f"  {name:<15s} {mean_ri[name]:.3f}  {bar}")
print()
print(f"  Most relied-on  : {top_two[0]}, {top_two[1]}")
print(f"  Least relied-on : {bottom_two[0]}, {bottom_two[1]}")
print()
print("DELETION EXPERIMENT")
if len(below):
    print(f"  CRISE-guided masking breaks rank-1 at {drop_pct:.0f}% of pixels")
    rnd_below = np.where(mean_random < RANK1_THRESH)[0]
    if len(rnd_below):
        rnd_pct = float(FINE_BUDGETS[rnd_below[0]] * 100)
        print(f"  Random masking breaks rank-1 at {rnd_pct:.0f}% of pixels")
        print(f"  CRISE efficiency gain: {rnd_pct / drop_pct:.1f}x fewer pixels needed")
print()
print("CROSS-PROBE STABILITY")
if n >= 2:
    print(f"  Mean pairwise cosine sim: {mean_cos:.3f} ± {std_cos:.3f}")
print()
print("FORENSIC INTERPRETATION")
print(f"  ArcFace's decision to identify you is disproportionately driven")
print(f"  by your {top_two[0].lower()} and {top_two[1].lower()}.")
print(f"  Masking just {drop_pct:.0f}% of your face (the most salient pixels)")
print(f"  is sufficient to break recognition.")
print(f"  This kind of evidence is invisible from ArcFace confidence scores alone.")
print("=" * 60)

# Save as text file
report_path = os.path.join(OUT_DIR, f"{YOUR_NAME}_forensics_report.txt")
import io, sys
# (just re-run print to file)
lines = [
    "=" * 60,
    f" CRISE-ID FORENSICS REPORT — {YOUR_NAME}",
    "=" * 60,
    f"  Gallery size           : {G_demo.shape[0]} identities",
    f"  Probe images analyzed  : {len(crise_results)}",
    f"  CRISE params           : N={N}, s={S}, p={P1}, tau={TAU}",
    "",
    "IDENTITY VERIFICATION",
    f"  ArcFace similarity     : {np.mean(clean_sims):.3f} ± {np.std(clean_sims):.3f}",
    f"  Rank-1 correct         : {n_correct}/{len(probe_results)}",
    "",
    "SALIENCY — REGION BREAKDOWN",
    *[f"  {name:<15s} {mean_ri[name]:.3f}" for name in sorted(mean_ri, key=mean_ri.get, reverse=True)],
    "",
    "CROSS-PROBE STABILITY",
    f"  Mean pairwise cosine sim: {mean_cos:.3f} ± {std_cos:.3f}" if n >= 2 else "  N/A",
]
with open(report_path, "w") as f:
    f.write("\n".join(lines))
print(f"\nReport saved: {report_path}")